---

<div align="center">
  <img src="https://raw.githubusercontent.com/devicons/devicon/master/icons/python/python-original.svg" width="80"/>
</div>

<h1 align="center">Chatbot RAG e agente (LangChain + Groq)</h1>

<h3 align="center">Ângelo Rosa</h3>

<div align="center">
  <img src="https://img.shields.io/badge/Python-3776AB?style=for-the-badge&logo=python&logoColor=white"/>
  <img src="https://img.shields.io/badge/Jupyter-F37626?style=for-the-badge&logo=jupyter&logoColor=white"/>
</div>

---

## 1. Contexto e objetivo

Dar uma camada conversacional sobre o que os módulos anteriores produziram: o banco do módulo 1 e
os embeddings e rótulos de sentimento do módulo 2.

Tudo é escrito aqui, passo a passo. O pacote `chatbot/` guarda esse mesmo código empacotado, e é o
que o `app.py` executa; este notebook não importa nada de lá, para que cada peça apareça por
extenso.

Duas formas de responder, e a diferença entre elas é o assunto da aula:

- **RAG puro**: busca os reviews mais parecidos com a pergunta e pede ao LLM uma resposta
  fundamentada **somente** neles. Reduz alucinação, mas não sabe contar nem calcular médias.
- **Agente**: o LLM decide sozinho entre consultar o banco em SQL (contagens, médias, agregações) e
  buscar nos textos (o que os clientes dizem).

**Stack gratuita:** os embeddings rodam localmente (módulo 2) e só a geração de texto vai para a
nuvem, com a Groq. Requer `GROQ_API_KEY` no `.env`, gratuita em console.groq.com.

In [1]:
import sqlite3
import textwrap

import numpy as np

from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_groq import ChatGroq

from config import (ANO_ALVO, CATEGORIA, DB_PATH, EMBEDDING_DIR, GROQ_API_KEY,
                    GROQ_MODEL, RAG_TOP_K, SENTIMENT_LABELS)
from utils.embedding import embed
from utils.sql import query_sql

if not GROQ_API_KEY:
    print("Sem GROQ_API_KEY no .env: as seções 2 e 3 rodam, da 4 em diante não.")
print("LLM:", GROQ_MODEL, "| dados:", CATEGORIA, ANO_ALVO)

c:\Users\PC\Documents\Projetos\amazon-reviews-chatbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LLM: openai/gpt-oss-120b | dados: All_Beauty 2021


## 2. O índice

É aqui que os módulos se encontram. A matriz de embeddings do módulo 2 fica lado a lado com o texto
do módulo 1 e o rótulo de sentimento previsto, todos alinhados pelo `review_id`.

O alinhamento é o detalhe que não pode falhar: `docs.loc[ids]` reordena a tabela na mesma ordem em
que os vetores foram gravados. Sem isso, a linha 10 da matriz não corresponderia ao review da linha
10 da tabela, e a busca devolveria textos aleatórios.

In [2]:
emb = np.load(f"{EMBEDDING_DIR}/emb.npy")
ids = np.load(f"{EMBEDDING_DIR}/review_ids.npy")

conn = sqlite3.connect(DB_PATH)
docs = query_sql(conn, """
    SELECT r.review_id, r.rating, r.title, r.text, s.sentiment_label
    FROM reviews r
    LEFT JOIN review_sentiment s ON s.review_id = r.review_id
""").set_index("review_id")
conn.close()

docs = docs.loc[ids].reset_index()          # mesma ordem da matriz de embeddings
docs["sentimento"] = docs["sentiment_label"].map(SENTIMENT_LABELS)

print(f"índice: {len(docs):,} reviews | {emb.shape[1]} dimensões")
docs.head(3)

índice: 111,851 reviews | 384 dimensões


,review_id,rating,title,text,sentiment_label,sentimento
0,1,5.0,exquisite,Beautiful brush collection.,2,positivo
1,2,5.0,Easy for beginners and update: bought 2 more,I'm very new at magnetic eyelashes and had bou...,2,positivo
2,3,4.0,I like eye patches,They are so relaxing and you can leave them on...,2,positivo


## 3. A busca por cosseno

A busca inteira são três linhas: embutir a pergunta com o **mesmo** modelo que embutiu os
documentos, multiplicar pela matriz e pegar os maiores. Como os vetores saem normalizados do módulo
2, o produto interno já é a similaridade de cosseno, e nenhum review precisa ser reembutido.

A consulta vai em inglês porque os reviews estão em inglês, e o modelo de embedding foi treinado
nessa língua.

In [3]:
def formatar(trechos):
    """Os reviews recuperados, no formato que vai para o prompt."""
    return "\n".join(
        f"- [nota {int(r.rating)} | {r.sentimento}] {r.title}: "
        f"{' '.join(str(r.text).split())[:220]}"
        for r in trechos.itertuples()
    )


def buscar(pergunta, k=RAG_TOP_K):
    similaridades = emb @ embed([pergunta], show_progress=False)[0]
    return docs.iloc[np.argsort(-similaridades)[:k]]


print(formatar(buscar("bad smell and cheap packaging", k=3)))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3296.91it/s]


- [nota 3 | negativo] Unsealed container and unsealed packaging: Not worth returning but came unsealed both the container and packaging. Looked used. Did smell good
- [nota 1 | negativo] Bad product: Smell won’t stay long too expensive
- [nota 3 | neutro] Smells good but not packaged properly: some of the product leaked out and made an oily mess trying to get it out of the package. Lost about an inch of the product in the packaging.


## 4. RAG puro

Recuperar e responder. O prompt do sistema amarra o modelo aos trechos recuperados: se eles não
bastarem, ele deve dizer isso em vez de inventar. É a diferença entre um chatbot que cita os dados
e um que alucina.

**A consulta precisa parecer com o texto procurado, não com uma pergunta sobre ele.** A busca
compara vetores de significado, então "poor quality, broke after a few days" encontra reviews que
dizem exatamente isso. Já "What do customers complain about the most?" traz o review *"Satisfied
customer: it really gets the job done"*, porque a frase que mais se parece com essa pergunta é uma
que fala de clientes satisfeitos. O vocabulário bate, o sentido não.

Vale a mesma observação da seção 3 sobre a língua: a consulta vai em inglês porque os reviews estão
em inglês. A resposta sai em português de qualquer forma, porque quem decide isso é o prompt do
sistema.

In [4]:
llm = ChatGroq(model=GROQ_MODEL, temperature=0, max_tokens=2048)

RAG_SYSTEM = ("Você é um analista de reviews. Responda em português, de forma objetiva, usando "
              "SOMENTE os trechos fornecidos. Se eles não bastarem, diga isso claramente.")


def responder_rag(pergunta, k=RAG_TOP_K):
    contexto = formatar(buscar(pergunta, k))
    resposta = llm.invoke([
        SystemMessage(RAG_SYSTEM),
        HumanMessage(f"Trechos de reviews:\n{contexto}\n\nPergunta: {pergunta}"),
    ])
    return resposta.content


print(textwrap.fill(responder_rag("poor quality, broke after a few days"), 96))

Os reviews indicam baixa qualidade e quebra rápida:  - “Poor quality: Broke really easily and
quick”   - “Poor quality: Broke the same day it came”   - “Poor quality: I purchased less than 1
month ago and already broken 😞”   - “Poor quality: It fell apart after a few uses”   - “Poor
quality: Not a great quality, would not buy again”


## 5. O agente e suas duas ferramentas

O RAG puro não sabe contar. Perguntado "quantos reviews negativos existem?", ele olha cinco trechos
e chuta. O agente resolve isso tendo duas ferramentas e decidindo qual usar:

- `consultar_sql`: consulta o banco, para contagens, médias e agregações.
- `busca_semantica`: a mesma busca da seção 3, para o que os clientes dizem.

Quem ensina o agente a escolher é o texto: a docstring da ferramenta e o prompt do sistema. Por isso
o prompt diz explicitamente para não procurar assunto com `LIKE` no SQL, que é o erro clássico.

As duas ferramentas são funções comuns com o decorador `@tool`. A conexão abre em modo somente
leitura (`mode=ro`), então uma consulta que tente escrever falha em vez de alterar o banco, e o
resultado é cortado em 30 linhas para não estourar o contexto do modelo.

In [5]:
@tool
def busca_semantica(pergunta: str) -> str:
    """Busca semântica nos textos dos reviews. Use para perguntas sobre o que os clientes dizem:
    reclamações, elogios e temas recorrentes."""
    return formatar(buscar(pergunta))


@tool
def consultar_sql(query: str) -> str:
    """Executa uma consulta SQL de leitura e devolve as primeiras linhas do resultado."""
    with sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True) as conn:   # só leitura
        return query_sql(conn, query).head(30).to_string(index=False)

SYSTEM = (
    "Você é o assistente de uma plataforma de análise de reviews da Amazon. Responda sempre em "
    f"português, de forma objetiva e fundamentada nos dados. O banco tem todos os reviews da "
    f"categoria {CATEGORIA} em {ANO_ALVO}. "
    "Para contagens, médias e agregações, use consultar_sql com UMA consulta só, que responda a "
    "pergunta inteira: para séries por período use GROUP BY strftime('%m', review_date), nunca "
    "uma consulta por mês. "
    "Para o que os clientes dizem (reclamações, elogios, temas), use busca_semantica UMA vez, "
    "nunca LIKE sobre text ou title. Não repita a mesma chamada de ferramenta. "
    "Tabelas: reviews(review_id, product_id, user_id, rating, title, text, review_date, "
    "helpful_vote, verified_purchase); products(product_id, title, main_category, average_rating, "
    "rating_number, price, store); review_sentiment(review_id, sentiment_label[0=negativo, "
    "1=neutro, 2=positivo], prob_positivo)."
)

agente = create_agent(llm, [busca_semantica, consultar_sql], system_prompt=SYSTEM)


def perguntar(pergunta):
    saida = agente.invoke({"messages": [{"role": "user", "content": pergunta}]})
    chamadas = [(c["name"], c.get("args", {})) for m in saida["messages"]
                for c in getattr(m, "tool_calls", []) or []]

    print("PERGUNTA:", pergunta)
    print(f"ferramentas: {len(chamadas)} chamada(s)")
    for nome, args in chamadas:                       # o que o agente de fato escreveu
        argumento = args.get("query") or args.get("pergunta", "")
        print(f"  {nome}: {' '.join(str(argumento).split())[:100]}")
    print("\nRESPOSTA:\n" + textwrap.fill(saida["messages"][-1].content, 96))

## 6. Perguntas de negócio

Três perguntas que exercitam caminhos diferentes: agregação simples, agregação temporal e busca nos
textos. Agora a saída mostra **qual** ferramenta o agente chamou, **quantas vezes** e **o que ele
escreveu**, que é onde está a parte instrutiva.

Repare no contraste. A primeira e a terceira saem em uma chamada cada, direto ao ponto. A segunda,
a série mensal, o agente repete: refaz a mesma consulta várias vezes, começando por versões com um
`JOIN products` que nada acrescenta, mesmo com o prompt pedindo uma consulta só com `GROUP BY`.

Duas lições para a aula. A primeira é que **o prompt orienta, não obriga**: o modelo decide, e às
vezes decide mal, ao custo de idas extras ao banco e à API. A segunda aparece no texto que ele manda
para a `busca_semantica`: ele pergunta *"quais são as reclamações mais comuns dos clientes?"*, que é
justamente a formulação que a seção 4 mostrou ser fraca, porque a busca compara a pergunta com o
texto dos reviews. Um agente melhor escreveria a consulta como uma frase de reclamação.

O número de chamadas muda de execução para execução, então a sua saída pode não bater exatamente
com a gravada aqui.

In [6]:
perguntar("Quantos reviews negativos, neutros e positivos existem?")

PERGUNTA: Quantos reviews negativos, neutros e positivos existem?
ferramentas: 1 chamada(s)
  consultar_sql: SELECT sentiment_label, COUNT(*) AS count FROM review_sentiment GROUP BY sentiment_label

RESPOSTA:
**Quantidade de reviews por sentimento (2021 – categoria All_Beauty)**    | Sentimento | Código
| Quantidade | |------------|--------|------------| | Negativo   | 0      | 29 937 | | Neutro
| 1      | 16 966 | | Positivo   | 2      | 64 948 |  Total de reviews analisados: **111 851**.
Esses valores foram obtidos diretamente da tabela **review_sentiment**, agrupando por
`sentiment_label`.


In [7]:
perguntar("Como a nota média variou mês a mês ao longo do ano?")

PERGUNTA: Como a nota média variou mês a mês ao longo do ano?
ferramentas: 5 chamada(s)
  consultar_sql: SELECT strftime('%m', r.review_date) AS mes, ROUND(AVG(r.rating), 2) AS media_nota FROM reviews r JO
  consultar_sql: SELECT strftime('%m', review_date) AS mes, ROUND(AVG(rating), 2) AS media_nota FROM reviews r JOIN p
  consultar_sql: SELECT DISTINCT main_category FROM products LIMIT 20;
  consultar_sql: SELECT review_date FROM reviews LIMIT 5;
  consultar_sql: SELECT strftime('%m', r.review_date) AS mes, ROUND(AVG(r.rating), 2) AS media_nota FROM reviews r JO

RESPOSTA:
**Variação da nota média (All_Beauty – 2021)**    | Mês | Nota média | |-----|------------| |
Jan | 3,91 | | Fev | 3,80 | | Mar | 3,81 | | Abr | 3,79 | | Mai | 3,75 | | Jun | 3,69 | | Jul |
3,71 | | Ago | 3,66 | | Set | 3,61 | | Out | 3,58 | | Nov | 3,58 | | Dez | 3,60 |  **Observações
objetivas**  - A nota média começou o ano em **3,91** (janeiro) e apresentou queda gradual até
**3,58** em outubro e novembro, o me

In [8]:
perguntar("Quais são as reclamações mais comuns dos clientes?")

PERGUNTA: Quais são as reclamações mais comuns dos clientes?
ferramentas: 1 chamada(s)
  busca_semantica: Quais são as reclamações mais comuns dos clientes nos reviews da categoria All_Beauty em 2021?

RESPOSTA:
**Reclamações mais comuns (conforme a análise semântica dos reviews da categoria *All_Beauty* em
2021)**    | Tipo de reclamação | Frequência (observada nos dados) | Exemplo de comentário |
|--------------------|----------------------------------|-----------------------| | **Pouca ou
nenhuma reclamação** | Muito baixa – a maioria dos trechos retornados são positivos e poucos
reviews apresentam sentimentos negativos. | “Me facino: Estoy facinada conneste producto”
(classificado como negativo, mas não contém crítica ao produto). | | **Problemas de
qualidade/uso** | Não foram identificados padrões recorrentes de queixas sobre qualidade,
eficácia ou uso do produto. | — | | **Embalagem/Entrega** | Não há indícios de reclamações
frequentes sobre embalagem ou prazo de entrega. | — | |

## 7. Da aula para a aplicação

O mesmo caminho está empacotado em `chatbot/`, em dois arquivos: `rag.py` (índice, busca e RAG
puro) e `agent.py` (ferramentas e agente). A interface é o `app.py`:

```
streamlit run app.py
```